## Improving all data with defined business rules

#### All Imports

In [103]:
import requests
import pandas as pd

##### Getting functions to read db:

In [104]:
%run "../functions/nb_function_read_db.ipynb"

In [ ]:
# Functions and data to calculate SLA and difference between two dates (in hours) unconsidering holidays
%run "../functions/nb_function_sla_calculation_holidays.ipynb"

In [121]:
# function to write df_jira_trusted in db_gold.parquet
%run "../functions/nb_function_write_db.ipynb"

In [106]:
# Getting data from silver parquet
df_jira = read_parquet(path="../../data/silver/db_silver.parquet")

In [107]:
display(df_jira)

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
0,JIRA-0001,Bug,Low,Open,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-08-02T14:55:05Z,None
1,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
2,JIRA-0003,Story,High,Open,adyna@fasttrack.com,u005,Adyna,2025-06-30T17:06:48Z,None
3,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
...,...,...,...,...,...,...,...,...,...
995,JIRA-0996,Task,Low,Done,francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
996,JIRA-0997,Task,Medium,Open,guilherme@fasttrack.com,u003,Guilherme Francisco,2026-02-30T25:61:00Z,not_a_date
997,JIRA-0998,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date
998,JIRA-0999,Task,High,Done,adyna@fasttrack.com,u005,Adyna,2026-02-30T25:61:00Z,not_a_date


In [108]:
# filtering df with "status" not like "Open"
df_jira_not_open = df_jira[df_jira["status"] != "Open"]
display(df_jira_not_open)

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
1,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
3,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
7,JIRA-0008,Story,High,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-04-07T22:55:41Z,2025-04-08T14:55:41Z
8,JIRA-0009,Story,Medium,Resolved,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-10-08T21:21:55Z,2025-10-12T10:21:55Z
...,...,...,...,...,...,...,...,...,...
993,JIRA-0994,Task,High,Done,francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
994,JIRA-0995,Bug,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date
995,JIRA-0996,Task,Low,Done,francinne@fasttrack.com,u004,Francinne,2026-02-30T25:61:00Z,not_a_date
997,JIRA-0998,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2026-02-30T25:61:00Z,not_a_date


In [109]:
# filtering timestamps_resolved_at not like a date and "not a date"
df_not_open_valid_data_resolved = df_jira_not_open.loc[df_jira_not_open["timestamps_resolved_at"].notna()]
df_not_open_valid_data_resolved = df_not_open_valid_data_resolved[df_not_open_valid_data_resolved["timestamps_resolved_at"] != "not_a_date"]
display(df_not_open_valid_data_resolved)

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
1,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
3,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
4,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
7,JIRA-0008,Story,High,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-04-07T22:55:41Z,2025-04-08T14:55:41Z
8,JIRA-0009,Story,Medium,Resolved,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-10-08T21:21:55Z,2025-10-12T10:21:55Z
...,...,...,...,...,...,...,...,...,...
982,JIRA-0983,Bug,Medium,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-12-04T08:01:14Z,2025-12-07T01:01:14Z
983,JIRA-0984,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-01-09T21:09:08Z,2025-01-14T22:09:08Z
985,JIRA-0986,Story,Low,Done,francinne@fasttrack.com,u004,Francinne,2025-02-09T05:45:27Z,2025-02-14T09:45:27Z
986,JIRA-0987,Task,Low,Done,guilherme@fasttrack.com,u003,Guilherme Francisco,2025-06-22T11:58:34Z,2025-06-28T01:58:34Z


In [110]:
# Reseting index
df_jira_trusted = df_not_open_valid_data_resolved.reset_index(drop=True)
display(df_jira_trusted)

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at
0,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09T12:39:26Z,2025-03-10T10:39:26Z
1,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22T20:24:58Z,2025-11-23T07:24:58Z
2,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06T01:08:44Z,2025-01-07T09:08:44Z
3,JIRA-0008,Story,High,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-04-07T22:55:41Z,2025-04-08T14:55:41Z
4,JIRA-0009,Story,Medium,Resolved,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-10-08T21:21:55Z,2025-10-12T10:21:55Z
...,...,...,...,...,...,...,...,...,...
799,JIRA-0983,Bug,Medium,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-12-04T08:01:14Z,2025-12-07T01:01:14Z
800,JIRA-0984,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-01-09T21:09:08Z,2025-01-14T22:09:08Z
801,JIRA-0986,Story,Low,Done,francinne@fasttrack.com,u004,Francinne,2025-02-09T05:45:27Z,2025-02-14T09:45:27Z
802,JIRA-0987,Task,Low,Done,guilherme@fasttrack.com,u003,Guilherme Francisco,2025-06-22T11:58:34Z,2025-06-28T01:58:34Z


In [111]:
# Casting fields to a correct datatypes
df_jira_trusted["timestamps_created_at"] = pd.to_datetime(
    df_jira_trusted["timestamps_created_at"],
    errors="coerce"
)

df_jira_trusted["timestamps_resolved_at"] = pd.to_datetime(
    df_jira_trusted["timestamps_resolved_at"],
    errors="coerce"
)

df_jira_trusted["id"] = df_jira_trusted["id"].astype("string")
df_jira_trusted["issue_type"] = df_jira_trusted["issue_type"].astype("string")
df_jira_trusted["priority"] = df_jira_trusted["priority"].astype("string")
df_jira_trusted["status"] = df_jira_trusted["status"].astype("string")
df_jira_trusted["assignee_email"] = df_jira_trusted["assignee_email"].astype("string")
df_jira_trusted["assignee_id"] = df_jira_trusted["assignee_id"].astype("string")
df_jira_trusted["assignee_name"] = df_jira_trusted["assignee_name"].astype("string")

In [112]:
# checking data types
df_jira_trusted.dtypes

id                             string[python]
issue_type                     string[python]
priority                       string[python]
status                         string[python]
assignee_email                 string[python]
assignee_id                    string[python]
assignee_name                  string[python]
timestamps_created_at     datetime64[ns, UTC]
timestamps_resolved_at    datetime64[ns, UTC]
dtype: object

In [113]:
# checking years from timestamps_resolved_at to get correct dates from Hollydays API
display(df_jira_trusted["timestamps_resolved_at"].dt.year.drop_duplicates())

0     2025
10    2026
Name: timestamps_resolved_at, dtype: int32

In [114]:
# checking years from timestamps_created_at to get correct dates from Hollydays API
display(df_jira_trusted["timestamps_created_at"].dt.year.drop_duplicates())

0     2025
10    2026
Name: timestamps_created_at, dtype: int32

In [115]:
# getting holidays of 2025 and 2026 (our data is from both year)
dt_feriados_2025 = obter_feriados_brasil(2025)
dt_feriados_2026 = obter_feriados_brasil(2026)

# getting hollydays of both years
dt_feriados = dt_feriados_2025 + dt_feriados_2026
dt_feriados

[datetime.date(2025, 1, 1),
 datetime.date(2025, 3, 4),
 datetime.date(2025, 4, 18),
 datetime.date(2025, 4, 20),
 datetime.date(2025, 4, 21),
 datetime.date(2025, 5, 1),
 datetime.date(2025, 6, 19),
 datetime.date(2025, 9, 7),
 datetime.date(2025, 10, 12),
 datetime.date(2025, 11, 2),
 datetime.date(2025, 11, 15),
 datetime.date(2025, 11, 20),
 datetime.date(2025, 12, 25),
 datetime.date(2026, 1, 1),
 datetime.date(2026, 2, 17),
 datetime.date(2026, 4, 3),
 datetime.date(2026, 4, 5),
 datetime.date(2026, 4, 21),
 datetime.date(2026, 5, 1),
 datetime.date(2026, 6, 4),
 datetime.date(2026, 9, 7),
 datetime.date(2026, 10, 12),
 datetime.date(2026, 11, 2),
 datetime.date(2026, 11, 15),
 datetime.date(2026, 11, 20),
 datetime.date(2026, 12, 25)]

In [116]:
# business_hour_resolution is a new column considering business day/hour with Brazil hollydays
df_jira_trusted["resolution_time_business_hour"] = df_jira_trusted.apply(
    lambda x: business_hour_without_holiday(
        x["timestamps_created_at"],
        x["timestamps_resolved_at"],
        dt_feriados
    ),
    axis=1
)

In [117]:
# creating fields with SLA Business Rules
df_jira_trusted["SLA_esperado"] = df_jira_trusted["priority"].map(sla_rules)

df_jira_trusted["SLA_atingido"] = "Violado"

df_jira_trusted.loc[
    df_jira_trusted["resolution_time_business_hour"] <= df_jira_trusted["SLA_esperado"],
    "SLA_atingido"
] = "Atendido"

In [118]:
df_jira_trusted

,id,issue_type,priority,status,assignee_email,assignee_id,assignee_name,timestamps_created_at,timestamps_resolved_at,resolution_time_business_hour,SLA_esperado,SLA_atingido
0,JIRA-0002,Bug,High,Resolved,francinne@fasttrack.com,u004,Francinne,2025-03-09 12:39:26+00:00,2025-03-10 10:39:26+00:00,11,24,Atendido
1,JIRA-0004,Story,High,Done,francinne@fasttrack.com,u004,Francinne,2025-11-22 20:24:58+00:00,2025-11-23 07:24:58+00:00,0,24,Atendido
2,JIRA-0005,Task,Medium,Resolved,francinne@fasttrack.com,u004,Francinne,2025-01-06 01:08:44+00:00,2025-01-07 09:08:44+00:00,33,72,Atendido
3,JIRA-0008,Story,High,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-04-07 22:55:41+00:00,2025-04-08 14:55:41+00:00,17,24,Atendido
4,JIRA-0009,Story,Medium,Resolved,gionni.lucio@fasttrack.com,u002,Gionni Lucio,2025-10-08 21:21:55+00:00,2025-10-12 10:21:55+00:00,51,72,Atendido
...,...,...,...,...,...,...,...,...,...,...,...,...
799,JIRA-0983,Bug,Medium,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-12-04 08:01:14+00:00,2025-12-07 01:01:14+00:00,40,72,Atendido
800,JIRA-0984,Task,Low,Resolved,matheus.malta@fasttrack.com,u001,Matheus Malta,2025-01-09 21:09:08+00:00,2025-01-14 22:09:08+00:00,74,120,Atendido
801,JIRA-0986,Story,Low,Done,francinne@fasttrack.com,u004,Francinne,2025-02-09 05:45:27+00:00,2025-02-14 09:45:27+00:00,106,120,Atendido
802,JIRA-0987,Task,Low,Done,guilherme@fasttrack.com,u003,Guilherme Francisco,2025-06-22 11:58:34+00:00,2025-06-28 01:58:34+00:00,120,120,Atendido


In [ ]:
# writing data to last DB of Medallion Architecture
write_db(df_jira_trusted,"../../data/gold/db_gold.parquet")

db_bronze.parquet gerado com sucesso!
